
# Hyperparam-Optimierung (v8) - Headless / No-Save

Ziel: Optimiere Hyperparameter strikt entlang der Pipeline-Logik (Training & Val-Split),
ohne Modelle/Scaler/Artefakte zu speichern und ohne Web-App.
Evaluation erfolgt ausschliesslich auf der im Speicher erzeugten Validierungsmenge (kein CSV-Fallback).


In [1]:

# Robust: Projektpfad in Jupyter ohne __file__
from pathlib import Path
import os, sys, tempfile, shutil, warnings

def _find_project_root():
    here = Path.cwd()
    sentinels = {"ML_Helpfunctions", "ML_Algorithms", "Input", "experiment", "config"}
    for p in [here, *list(here.parents)[:4]]:
        try:
            contents = {c.name for c in p.iterdir() if c.is_dir()}
        except Exception:
            contents = set()
        if sentinels & contents:
            return p
    return here

PROJECT_ROOT = _find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

# User Windows paths (optional)
WIN_ROOT = r"C:\Users\ericg\Documents\Mechatronik M Sc\6. Semster\MA\Dev_Ma\ML_Edge_Device"
for p in [WIN_ROOT, rf"{WIN_ROOT}\experiment", rf"{WIN_ROOT}\ML_Helpfunctions", rf"{WIN_ROOT}\ML_Algorithms"]:
    try:
        if p and Path(p).exists() and p not in sys.path:
            sys.path.append(p)
    except Exception:
        pass

warnings.filterwarnings("ignore")
print("PROJECT_ROOT =", PROJECT_ROOT)


PROJECT_ROOT = c:\Users\ericg\Documents\Mechatronik M Sc\6. Semster\MA\Dev_Ma\ML_Edge_Device


In [2]:

# --- Project paths (Windows root preferred) ---
import sys, os
from pathlib import Path

# Try to lock to your Windows project root (if this path exists on the local machine)
WIN_ROOT = Path(r"C:\Users\ericg\Documents\Mechatronik M Sc\6. Semster\MA\Dev_Ma\ML_Edge_Device")
if WIN_ROOT.exists():
    PROJECT_ROOT = WIN_ROOT
else:
    # Notebook/Script fallback
    try:
        PROJECT_ROOT = Path(__file__).resolve().parent
    except NameError:
        PROJECT_ROOT = Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

# --- Config imports ---
try:
    from config.config_general import CONFIG_PATH, CONFIG_LOAD_ARTIFACTS, MQTT_CONFIG  # type: ignore
except ModuleNotFoundError:
    try:
        from config_general import CONFIG_PATH, CONFIG_LOAD_ARTIFACTS, MQTT_CONFIG  # type: ignore
    except ModuleNotFoundError:
        # Minimal fallback so that CSV exports have a destination
        CONFIG_PATH = {"paths": {"output": str(PROJECT_ROOT / "Output")}}
        CONFIG_LOAD_ARTIFACTS = {}
        MQTT_CONFIG = {}
        
# --- pipeline_utils import with fallback ---
try:
    from ML_Helpfunctions import pipeline_utils as PU  # type: ignore
except ModuleNotFoundError:
    import importlib
    PU = importlib.import_module('pipeline_utils')


In [3]:

import numpy as np
import pandas as pd
import optuna
from datetime import datetime
import gc

ALGORITHM = "lstm"
LEVEL = "medium"
DEFAULT_LAGS = 20
DEFAULT_H = 8
N_TRIALS = 50
SEED = 42
OBJECTIVE = "mae_avg"

QUANT_MODES = ["no-quant"]  # kein Quantisierungslauf in der Optimierung


In [4]:

def _try_import(module, name):
    import importlib
    m = importlib.import_module(module)
    return getattr(m, name)

# LSTM Trainer
try:
    LSTMTrainer = _try_import("ML_Algorithms.LSTM.lstm_train", "LSTMTrainer")
except Exception:
    from lstm_train import LSTMTrainer

# DataPipeline3D
try:
    from ML_Helpfunctions.Load_Prepare_Data import DataPipeline3D
except Exception:
    from Load_Prepare_Data import DataPipeline3D

# pipeline utils
try:
    from ML_Helpfunctions import pipeline_utils as PU
except Exception:
    import pipeline_utils as PU

# Optional config builder from experiment pipeline
_build_cfg = None
try:
    from experiment_pipeline_lag_horizon import build_training_config as _build_cfg
except Exception:
    try:
        from experiment_pipeline_multiconfig import build_training_config as _build_cfg
    except Exception:
        _build_cfg = None


In [5]:

def suggest_params_level(trial, algo: str, level: str) -> dict:
    a, L = (algo or "").lower(), (level or "").lower()
    P = {}

    # ---------- LSTM ----------
    if a == "lstm":
        if L == "simple":
            P["num_layers"] = 1
            P["units1"] = trial.suggest_int("units1", 32, 64, step=16)
            P["dropout"] = trial.suggest_float("dropout", 0.00, 0.15)
            P["learning_rate"] = trial.suggest_float("learning_rate", 5e-4, 2e-3, log=True)
            P["batch_size"] = trial.suggest_categorical("batch_size", [32, 64])
            P["epochs"] = trial.suggest_int("epochs", 15, 35)
        elif L == "medium":
            P["num_layers"] = 2
            P["units1"] = trial.suggest_int("units1", 64, 128, step=32)
            P["units2"] = trial.suggest_int("units2", 32, 96, step=32)
            P["dropout"] = trial.suggest_float("dropout", 0.05, 0.30)
            P["learning_rate"] = trial.suggest_float("learning_rate", 2e-4, 1e-3, log=True)
            P["batch_size"] = trial.suggest_categorical("batch_size", [32, 64, 96])
            P["epochs"] = trial.suggest_int("epochs", 25, 60)
        else:  # high
            P["num_layers"] = trial.suggest_int("num_layers", 3, 4)
            P["units1"] = trial.suggest_int("units1", 128, 256, step=64)
            P["units2"] = trial.suggest_int("units2", 64, 192, step=64)
            P["units3"] = trial.suggest_int("units3", 64, 128, step=32)
            if P["num_layers"] == 4:
                P["units4"] = trial.suggest_int("units4", 32, 96, step=32)
            P["dropout"] = trial.suggest_float("dropout", 0.10, 0.40)
            P["learning_rate"] = trial.suggest_float("learning_rate", 5e-5, 7e-4, log=True)
            P["batch_size"] = trial.suggest_categorical("batch_size", [32, 64, 96, 128])
            P["epochs"] = trial.suggest_int("epochs", 35, 80)

    # ---------- 1D-CNN ----------
    elif a == "cnn1d":
        if L == "simple":
            P["conv_blocks"] = 1
            P["filters"] = trial.suggest_int("filters", 16, 32, step=8)
            P["kernel_size"] = trial.suggest_int("kernel_size", 3, 5, step=2)
            P["dropout"] = trial.suggest_float("dropout", 0.00, 0.15)
            P["learning_rate"] = trial.suggest_float("learning_rate", 5e-4, 2e-3, log=True)
        elif L == "medium":
            P["conv_blocks"] = 2
            P["filters"] = trial.suggest_int("filters", 32, 64, step=16)
            P["kernel_size"] = trial.suggest_int("kernel_size", 3, 7, step=2)
            P["dropout"] = trial.suggest_float("dropout", 0.05, 0.30)
            P["learning_rate"] = trial.suggest_float("learning_rate", 2e-4, 1e-3, log=True)
        else:  # high
            P["conv_blocks"] = trial.suggest_int("conv_blocks", 3, 4)
            P["filters"] = trial.suggest_int("filters", 64, 128, step=32)
            P["kernel_size"] = trial.suggest_int("kernel_size", 5, 11, step=2)
            P["dropout"] = trial.suggest_float("dropout", 0.10, 0.40)
            P["learning_rate"] = trial.suggest_float("learning_rate", 5e-5, 7e-4, log=True)

    # ---------- Random Forest ----------
    elif a == "random_forest":
        if L == "simple":
            P["n_estimators"] = trial.suggest_int("n_estimators", 80, 180, step=20)
            P["max_depth"] = trial.suggest_int("max_depth", 8, 14)
            P["min_samples_split"] = trial.suggest_int("min_samples_split", 4, 10)
            P["min_samples_leaf"] = trial.suggest_int("min_samples_leaf", 2, 5)
            P["max_features"] = 1.0
        elif L == "medium":
            P["n_estimators"] = trial.suggest_int("n_estimators", 200, 350, step=25)
            P["max_depth"] = trial.suggest_int("max_depth", 14, 28)
            P["min_samples_split"] = trial.suggest_int("min_samples_split", 2, 6)
            P["min_samples_leaf"] = trial.suggest_int("min_samples_leaf", 1, 3)
            P["max_features"] = 0.8
        else:
            P["n_estimators"] = trial.suggest_int("n_estimators", 350, 600, step=50)
            P["max_depth"] = trial.suggest_int("max_depth", 28, 60)
            P["min_samples_split"] = trial.suggest_int("min_samples_split", 2, 4)
            P["min_samples_leaf"] = trial.suggest_int("min_samples_leaf", 1, 2)
            P["max_features"] = "sqrt"

    # ---------- XGBoost / Light_XGBOOST ----------
    elif a in ("xgboost","light_xgboost"):
        if L == "simple":
            P.update({
                "n_estimators": trial.suggest_int("n_estimators", 200, 500, step=50),
                "max_depth": trial.suggest_int("max_depth", 3, 4),
                "learning_rate": trial.suggest_float("learning_rate", 0.03, 0.12, log=True),
                "subsample": trial.suggest_float("subsample", 0.8, 1.0),
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.8, 1.0),
                "min_child_weight": trial.suggest_int("min_child_weight", 2, 5),
                "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 8.0),
                "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 1.0),
                "gamma": trial.suggest_float("gamma", 0.0, 2.0),
            })
        elif L == "medium":
            P.update({
                "n_estimators": trial.suggest_int("n_estimators", 500, 900, step=50),
                "max_depth": trial.suggest_int("max_depth", 4, 6),
                "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.08, log=True),
                "subsample": trial.suggest_float("subsample", 0.7, 0.95),
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 0.95),
                "min_child_weight": trial.suggest_int("min_child_weight", 1, 6),
                "reg_lambda": trial.suggest_float("reg_lambda", 2.0, 15.0),
                "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 3.0),
                "gamma": trial.suggest_float("gamma", 0.0, 6.0),
            })
        else:
            P.update({
                "n_estimators": trial.suggest_int("n_estimators", 900, 1200, step=50),
                "max_depth": trial.suggest_int("max_depth", 6, 9),
                "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.05, log=True),
                "subsample": trial.suggest_float("subsample", 0.6, 0.9),
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 0.9),
                "min_child_weight": trial.suggest_int("min_child_weight", 1, 8),
                "reg_lambda": trial.suggest_float("reg_lambda", 5.0, 20.0),
                "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 5.0),
                "gamma": trial.suggest_float("gamma", 0.0, 10.0),
            })

    # ---------- SVM ----------
    elif a == "svm":
        if L == "simple":
            P["svm_kernel"] = "linear"
            P["C"] = trial.suggest_float("C", 0.05, 10.0, log=True)
            P["epsilon"] = trial.suggest_float("epsilon", 0.01, 0.20, log=True)
            P["max_iter"] = trial.suggest_int("max_iter", 10000, 15000, step=5000)
        elif L == "medium":
            P["svm_kernel"] = "linear"
            P["C"] = trial.suggest_float("C", 10.0, 100.0, log=True)
            P["epsilon"] = trial.suggest_float("epsilon", 0.05, 0.30, log=True)
            P["max_iter"] = trial.suggest_int("max_iter", 15000, 25000, step=5000)
        else:
            P["svm_kernel"] = "rbf"
            P["C"] = trial.suggest_float("C", 1.0, 50.0, log=True)
            P["gamma"] = trial.suggest_float("gamma", 1e-3, 0.3, log=True)
            P["epsilon"] = trial.suggest_float("epsilon", 0.01, 0.20, log=True)
            P["max_iter"] = trial.suggest_int("max_iter", 20000, 40000, step=5000)

    # ---------- Ridge / Lasso ----------
    elif a == "ridge":
        if L == "simple":
            P["alpha"] = trial.suggest_float("alpha", 0.05, 3.0, log=True)
        elif L == "medium":
            P["alpha"] = trial.suggest_float("alpha", 0.01, 8.0, log=True)
        else:
            P["alpha"] = trial.suggest_float("alpha", 0.005, 20.0, log=True)

    elif a == "lasso":
        if L == "simple":
            P["alpha"] = trial.suggest_float("alpha", 0.05, 3.0, log=True)
        elif L == "medium":
            P["alpha"] = trial.suggest_float("alpha", 0.01, 8.0, log=True)
        else:
            P["alpha"] = trial.suggest_float("alpha", 0.005, 20.0, log=True)

    else:
        raise ValueError(f"Algorithm '{algo}' not supported in suggest_params.")

    return P


In [6]:

# --- Robust dispatcher to handle different suggest_params signatures ---
import inspect

def _dispatch_suggest_params(trial, algo: str, level: str):
    """
    Prefer suggest_params_level(trial, algo, level).
    Fallback to suggest_params(trial, algo, level) or suggest_params(trial, algo).
    """
    # 1) Prefer the explicit level-aware function
    f = globals().get("suggest_params_level", None)
    if callable(f):
        return f(trial, algo, level)

    # 2) Fall back to a generic suggest_params
    g = globals().get("suggest_params", None)
    if not callable(g):
        raise RuntimeError("Neither suggest_params_level nor suggest_params found.")

    try:
        sig = inspect.signature(g)
        n = len(sig.parameters)
    except Exception:
        try:
            n = g.__code__.co_argcount
        except Exception:
            n = 0

    if n >= 3:
        return g(trial, algo, level)
    elif n == 2:
        return g(trial, algo)
    else:
        raise RuntimeError("suggest_params() has unsupported signature.")


In [7]:

from pathlib import Path

def build_config(algorithm:str, level:str, *, lags:int, horizon:int, run_id:str):
    algo = algorithm.lower()
    folder_flag = algo.upper()

    if _build_cfg is not None:
        cfg = _build_cfg(algorithm=algo, level=level, horizon=int(horizon), folder_flag=folder_flag, quant_modes=QUANT_MODES)
        cfg["lags"] = int(lags)
    else:
        cfg = {
            "algorithm": algo, "model_name": algo, "level": level,
            "run_id": run_id, "time_stamp": run_id,
            "dataset": "mqtt_data_filtered.csv",
            "lags": int(lags), "horizon": int(horizon),
            "validation_fraction": 0.2,
            "scale_target": True, "scale_other_features": True,
            "scaler_type": "minmax",
            "use_early_stopping": True, "use_reduce_lr_on_plateau": True,
            "loss": "mse", "metrics": ["mae"],
            "quantization_enabled": False, "edge_device": False, "enable_edge": False,
            "start_web": False, "start_web_app": False,
            "paths": {}
        }

    tmpdir = Path(tempfile.mkdtemp(prefix="HPOv8_")).resolve()
    input_dir = Path(WIN_ROOT, "Input", "Input_Data") if Path(WIN_ROOT, "Input", "Input_Data").exists() else (PROJECT_ROOT / "Input" / "Input_Data")
    cfg["paths"] = {
        "Input_Data": str(input_dir),
        "input_data": str(input_dir),
        "Output": str(tmpdir),
        "run_dir": str(tmpdir),
        "Models": str(tmpdir / "Models"),
        "Error_Metrics": str(tmpdir / "Error_Metrics"),
        "Scalers": str(tmpdir / "Scalers"),
        "Model_Structures": str(tmpdir / "Model_Structures"),
        "Loss_Plots": str(tmpdir / "Loss_Plots"),
        "Prediction_Data": str(tmpdir / "Prediction_Data"),
    }

    cfg["no_save"] = True
    # tree-based scaler override
    if algo in ("random_forest", "xgboost", "light_xgboost"):
        cfg["scale_other_features"] = False
        cfg["scale_target"] = False

    cfg["headless"] = True
    return cfg

    cfg["quant_modes"] = QUANT_MODES


In [20]:
from Load_Prepare_Data import DataPipeline3D, DataPipeline2D

def run_training_in_memory(algorithm: str, level: str, *, lags: int, horizon: int, params: dict | None = None):
    run_id = datetime.now().strftime("%Y-%m-%d_%H%M%S_%f")[:-3] + "_train"

    # 1) Basiskonfig bauen (funktion 'build_config' muss im Notebook definiert sein)
    cfg = build_config(algorithm, level, lags=lags, horizon=horizon, run_id=run_id)

    # 2) Top-Level-Parameter übernehmen
    if params:
        cfg.update(params)

    # 3) Baum-Methoden: Scaler AUS
    algo_l = (algorithm or "").lower()
    if algo_l in ("random_forest", "xgboost", "light_xgboost"):
        cfg["scale_other_features"] = False
        cfg["scale_target"] = False
        # Falls verschachtelte Modell-Parameter geführt werden
        if "model_params" in cfg and isinstance(cfg["model_params"], dict) and isinstance(params, dict):
            for k, v in params.items():
                if k not in ("scale_target", "scale_other_features"):
                    cfg["model_params"][k] = v

    # 4) Trainer auflösen (erst Discovery, dann statisches Mapping)
    Trainer = None
    try:
        if "TRAINERS_FOUND" in globals() and algo_l in TRAINERS_FOUND:
            Trainer = TRAINERS_FOUND[algo_l]
    except Exception:
        pass

    if Trainer is None:
        try:
            mod, cls, _ = TRAINER_MAP[algo_l]
            Trainer = getattr(import_module(mod), cls)
        except Exception as e:
            raise ValueError(f"Algorithm '{algorithm}' not supported or trainer not found: {e}")

    # 5) Trainieren (ohne Artefakte zu speichern)
    try:
        trainer = Trainer(config=cfg, folder_flag=algorithm.upper())
    except TypeError:
        trainer = Trainer(config=cfg)

    out = trainer.run(save_artifacts=False)

    # 6) Artefakte robust extrahieren
    if isinstance(out, tuple) and len(out) >= 4:
        model, scaler, y_scaler, features = out[:4]
    else:
        model    = getattr(trainer, "model", None)
        scaler   = getattr(trainer, "scaler", None)
        y_scaler = getattr(trainer, "y_scaler", None)
        features = getattr(trainer, "feature_list", None)

    if isinstance(features, dict):
        features = features.get("all", [])
    features = list(features or [])

    # 7) Richtige Pipeline wählen: DL -> 3D, klassische Modelle -> 2D
    if algo_l in ("lstm", "cnn1d"):
        pipe = DataPipeline3D(cfg)
    else:
        pipe = DataPipeline2D(cfg)

    X_train, y_train = pipe.prepare_training_data()

    # Konsistente Dtypen + Speicherlayout
    X_train = np.asarray(X_train, dtype=np.float32, order="C")
    y_train = np.asarray(y_train, dtype=np.float32, order="C")

    # 8) Chronologischen Validierungs-Split erzeugen
    X_fit, y_fit, X_val, y_val = PU.create_timeseries_validation_split(X_train, y_train, cfg)

    X_fit = np.asarray(X_fit, dtype=np.float32, order="C") if X_fit is not None else None
    y_fit = np.asarray(y_fit, dtype=np.float32, order="C") if y_fit is not None else None
    X_val = np.asarray(X_val, dtype=np.float32, order="C") if X_val is not None else None
    y_val = np.asarray(y_val, dtype=np.float32, order="C") if y_val is not None else None

    return {
        "config": cfg,
        "model": model,
        "scaler": scaler,
        "y_scaler": y_scaler,
        "features": features,
        "X_fit": X_fit, "y_fit": y_fit, "X_val": X_val, "y_val": y_val,
    }


def _inverse_y(pred_scaled, true_scaled, y_scaler):
    """Inverse-Skalierung robust für (n, h)-Arrays; wenn kein y_scaler: Identität."""
    if y_scaler is None:
        return pred_scaled, true_scaled
    ps = pred_scaled.reshape(-1, 1)
    ts = true_scaled.reshape(-1, 1)
    pred = y_scaler.inverse_transform(ps).reshape(pred_scaled.shape)
    true = y_scaler.inverse_transform(ts).reshape(true_scaled.shape)
    return pred, true


def evaluate_on_validation(art, objective="mae_avg"):
    """Einheitliche Validierungsbewertung für 2D und 3D Modelle."""
    X_val, y_val = art["X_val"], art["y_val"]
    if X_val is None or y_val is None or len(X_val) == 0:
        raise RuntimeError("No validation data available.")

    X_val = np.ascontiguousarray(X_val, dtype=np.float32)
    y_val = np.ascontiguousarray(y_val, dtype=np.float32)

    model = art["model"]
    try:
        y_pred_scaled = model.predict(X_val, verbose=0)  # Keras-Modelle
    except TypeError:
        y_pred_scaled = model.predict(X_val)             # Sklearn/LightGBM

    y_pred, y_true = _inverse_y(np.asarray(y_pred_scaled), np.asarray(y_val), art["y_scaler"])

    if y_true.ndim == 1:
        y_true = y_true[:, None]
    if y_pred.ndim == 1:
        y_pred = y_pred[:, None]

    metrics = PU.evaluate_all_metrics(y_true=y_true, y_pred=y_pred, y_train=None, horizon=y_true.shape[1])

    if objective == "mae_avg":
        target = float(np.nanmean(metrics.get("mae", np.nan)))
    elif objective == "mae_h1":
        mae_arr = np.asarray(metrics.get("mae"))
        target = float(mae_arr[0]) if mae_arr.size else float("nan")
    else:
        target = float(np.nanmean(metrics.get("mae", np.nan)))

    return target, metrics


In [10]:

def algorithm_to_folder(name_or_flag: str) -> str:
    n = (name_or_flag or "").lower()
    if "light_xgboost" in n: return "Light_XGBOOST"
    if "lstm" in n: return "LSTM"
    if "cnn" in n: return "CNN1D"
    if "xgb" in n: return "XGBOOST"
    if "random_forest" in n: return "Random_Forest"
    if "ridge" in n or "lasso" in n: return "RIDGE_LASSO"
    if "svm" in n: return "SVM"
    return name_or_flag.upper() or "MODEL"


In [11]:
def _cleanup_backend():
    try:
        import tensorflow as tf
        try:
            tf.keras.backend.clear_session()
        except Exception:
            pass
    except Exception:
        pass
    gc.collect()


def suggest_params(trial: optuna.Trial, algorithm:str):
    algo = algorithm.lower()
    p = {}
    if algo == "lstm":
        p["lstm_units"]    = trial.suggest_int("lstm_units", 32, 128, step=32)
        p["dense_units"]   = trial.suggest_int("dense_units", 16, 64, step=16)
        p["dropout"]       = trial.suggest_float("dropout", 0.0, 0.4, step=0.1)
        p["learning_rate"] = trial.suggest_float("learning_rate", 1e-4, 5e-3, log=True)
        p["batch_size"]    = trial.suggest_categorical("batch_size", [32, 64, 128])
        p["epochs"]        = 30
    else:
        raise ValueError(f"Algorithm '{algorithm}' not supported in suggest_params.")
    return p

def objective_optuna(trial: optuna.Trial):
    params = _dispatch_suggest_params(trial, ALGORITHM, LEVEL)
    LAGS = DEFAULT_LAGS
    H    = DEFAULT_H
    try:
        art = run_training_in_memory(ALGORITHM, LEVEL, lags=LAGS, horizon=H, params=params)
        target, _ = evaluate_on_validation(art, objective=OBJECTIVE)
        trial.set_user_attr("val_target", target)
        trial.set_user_attr("lags", LAGS)
        trial.set_user_attr("horizon", H)
        return target
    except Exception as e:
        msg = str(e)
        print(f"[WARN] Trial {trial.number} failed once: {msg}")
        # retry once for low-level errors
        if ('listobject.c' in msg) or ('bad argument to internal function' in msg):
            _cleanup_backend()
            try:
                art = run_training_in_memory(ALGORITHM, LEVEL, lags=LAGS, horizon=H, params=params)
                target, _ = evaluate_on_validation(art, objective=OBJECTIVE)
                trial.set_user_attr("val_target", target)
                trial.set_user_attr("lags", LAGS)
                trial.set_user_attr("horizon", H)
                return target
            except Exception as e2:
                print(f"[WARN] Trial {trial.number} pruned after retry: {e2}")
                _cleanup_backend()
                raise optuna.TrialPruned()
        else:
            _cleanup_backend()
            raise optuna.TrialPruned()


In [12]:

# sampler = optuna.samplers.TPESampler(seed=SEED, multivariate=True)
# pruner  = optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=0)

# study_name = f"opt_{ALGORITHM}_{LEVEL}_v8_no_save"
# study = optuna.create_study(direction="minimize", study_name=study_name, sampler=sampler, pruner=pruner)
# print("Starte Studie:", study_name, "| Trials:", N_TRIALS)

# study.optimize(objective_optuna, n_trials=N_TRIALS, gc_after_trial=True, show_progress_bar=True)

# completed = [t for t in study.trials if t.state.name == "COMPLETE"]
# if completed:
#     best = study.best_trial
#     print("\nBeste Trial:")
#     print("  value     =", best.value)
#     print("  params    =", best.params)
#     print("  attrs     =", {k: best.user_attrs.get(k) for k in ['val_target','lags','horizon'] if k in best.user_attrs})
# else:
#     print("\nKeine Trials erfolgreich abgeschlossen (alle gepruned/failed).")

# # Export
# from pathlib import Path
# df_trials = study.trials_dataframe(attrs=("number","value","state","params","user_attrs"))
# out_csv = Path("HPOv8_results.csv").resolve()
# df_trials.to_csv(out_csv, index=False, encoding="utf-8")
# print("Gespeichert:", out_csv)

# try:
#     import tensorflow as tf
#     tf.keras.backend.clear_session()
# except Exception:
#     pass
# gc.collect()



## Hinweise
- Keine Speicherung: `save_artifacts=False` und Temp-Output verhindern persistente Dateien.
- Eval: nur `X_val / y_val`, erzeugt via `DataPipeline3D` und `create_timeseries_validation_split`.
- Anpassbar: `suggest_params`, `OBJECTIVE`, `DEFAULT_LAGS`, `DEFAULT_H`, `N_TRIALS`.
- Erweiterbar: Weitere Algorithmen analog zu LSTM ergänzen.



## Multi-Model Loop
Laufe die Optimierung nacheinander fuer alle verfügbaren Modelle (Trainer-Klassen werden dynamisch gesucht).
Ergebnisse werden pro Modell in eine eigene CSV geschrieben **(nur Optuna-Resultate, keine Artefakte)** und am Ende zu einer Sammel-Tabelle zusammengeführt.


In [ ]:

# === Trainer-Klassen je Algorithmus (wie in experiment_pipeline_lag_horizon.py) ===
TRAINER_MAP = {
    "lstm": ("ML_Algorithms.LSTM.lstm_train", "LSTMTrainer", "LSTM"),
    "cnn1d": ("ML_Algorithms.CNN1D.cnn1d_train", "CNN1DTrainer", "CNN1D"),
    "random_forest": ("ML_Algorithms.Random_Forest.rf_train", "RandomForestTrainer", "Random_Forest"),
    "xgboost": ("ML_Algorithms.XGBOOST.xgboost_train", "XGBoostTrainer", "XGBOOST"),
    "light_xgboost": ("ML_Algorithms.Light_XGBOOST.light_xgboost_train", "LightXGBoostTrainer", "Light_XGBOOST"),
    "ridge": ("ML_Algorithms.RIDGE.ridge_lasso_train", "RidgeLassoTrainer", "RIDGE_LASSO"),
    "svm": ("ML_Algorithms.SVM.svm_train", "SVMTrainer", "SVM"),
}
def _try_import(module: str, attr: str):
    import importlib
    m = importlib.import_module(module)
    return getattr(m, attr)

def discover_trainers():
    found = {}
    for algo, (mod, cls, _) in TRAINER_MAP.items():
        try:
            Trainer = _try_import(mod, cls)
            found[algo] = Trainer
        except Exception as e:
            # silently skip if module/class missing
            continue
    return found

TRAINERS_FOUND = discover_trainers()
print("Gefundene Trainer (mapping-basiert):", list(TRAINERS_FOUND.keys()))


Gefundene Trainer (mapping-basiert): ['lstm', 'cnn1d', 'random_forest', 'xgboost', 'light_xgboost', 'ridge', 'lasso', 'svm']


In [14]:

# --- Fallback globals for optimization loop ---
try:
    ALGORITHMS_ALL
except NameError:
    ALGORITHMS_ALL = ["lstm","cnn1d","random_forest","xgboost","light_xgboost","ridge","lasso","svm"]

def _ensure_trainers_found():
    # Ensure TRAINERS_FOUND exists by discovering from mapping if needed.
    global TRAINERS_FOUND
    try:
        TRAINERS_FOUND
        return TRAINERS_FOUND
    except NameError:
        pass
    # Prefer discover_trainers() if available
    try:
        TRAINERS_FOUND = discover_trainers()
        return TRAINERS_FOUND
    except Exception:
        pass
    # Fallback: try to import from TRAINER_MAP
    TRAINERS_FOUND = {}
    try:
        import importlib
        for algo, (mod, cls, _) in TRAINER_MAP.items():
            try:
                m = importlib.import_module(mod)
                Trainer = getattr(m, cls)
                TRAINERS_FOUND[algo] = Trainer
            except Exception:
                continue
    except Exception:
        TRAINERS_FOUND = {}
    return TRAINERS_FOUND


In [15]:

# -- Provide a simple factory that binds (algo, level) and reuses objective_optuna --
def make_objective_for_algorithm(algo: str, level: str):
    """
    Wraps the global objective_optuna(trial) by setting ALGORITHM and LEVEL
    before each trial. This keeps all logic centralized in objective_optuna.
    """
    def _obj(trial):
        globals()["ALGORITHM"] = algo
        globals()["LEVEL"] = level
        return objective_optuna(trial)
    return _obj


In [23]:

# Master-Loop: alle gefundenen Modelle optimieren
def optimize_all_models(algorithms=None, levels=("simple","medium","high"), n_trials=50, seed=SEED):
    TRAINERS = _ensure_trainers_found()
    try:
        algos_all = ALGORITHMS_ALL
    except NameError:
        algos_all = ["lstm","cnn1d","random_forest","xgboost","light_xgboost","ridge","lasso","svm"]

    algos = algorithms or [a for a in algos_all if a in TRAINERS]
    if not algos:
        print("Keine passenden Trainer gefunden.")
        return None

    results = []
    for algo in algos:
        for level in levels:
            print(f"\n=== Optimize: {algo.upper()} | LEVEL: {level.upper()} ===")
            sampler = optuna.samplers.TPESampler(seed=seed, multivariate=True)
            pruner  = optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=0)

            study_name = f"opt_{algo}_{level}_v8_no_save"
            study = optuna.create_study(direction="minimize", study_name=study_name, sampler=sampler, pruner=pruner)

            obj = make_objective_for_algorithm(algo, level)
            study.optimize(obj, n_trials=n_trials or 50, gc_after_trial=True, show_progress_bar=True)

            completed = [t for t in study.trials if t.state.name == "COMPLETE"]
            if completed:
                best = study.best_trial
                print(f"Best [{algo}|{level}]:", best.value, best.params)
            else:
                print("Keine erfolgreichen Trials.")

            # CSV pro Modell/Level
            df_trials = study.trials_dataframe(attrs=("number","value","state","params","user_attrs"))
            csv_path = Path(CONFIG_PATH["paths"]["output"]) / f"HPOv8_results_{algo}_{level}.csv"
            df_trials.to_csv(csv_path, index=False, encoding="utf-8")
            print("Gespeichert:", csv_path)

            # Sammeln fuer Gesamtübersicht
            df_trials.insert(0, "algorithm", algo)
            df_trials.insert(1, "level", level)
            results.append(df_trials)

    if results:
        import pandas as pd
        df_all = pd.concat(results, ignore_index=True)
        all_path = Path(CONFIG_PATH["paths"]["output"]) / "HPOv8_results_ALL.csv"
        df_all.to_csv(all_path, index=False, encoding="utf-8")
        print("\nSammel-Ergebnisse:", all_path)
        return df_all
    return None


In [24]:

# Ausführen (nutzt automatisch alle gefundenen Trainer)
df_all = optimize_all_models()


[I 2025-09-03 03:12:06,986] A new study created in memory with name: opt_lstm_simple_v8_no_save



=== Optimize: LSTM | LEVEL: SIMPLE ===


  0%|          | 0/50 [00:00<?, ?it/s]

2025-09-03 03:12:06,999 - INFO - --- 🚀 Starting lstm_simple Training Pipeline ---
2025-09-03 03:12:06,999 - INFO - 
Step 1: Preparing training data...
2025-09-03 03:12:07,045 - INFO - Ordne Features neu an, um Target 'group4-2_s6_volumetricflowrate' an Position 0 zu platzieren.
2025-09-03 03:12:07,075 - INFO - Die zu speichernde Feature-Liste für die Inferenz enthält 46 Spalten.
2025-09-03 03:12:07,076 - INFO - Data preparation complete. Features: 46, X_train shape: (2609, 20, 46)
2025-09-03 03:12:07,076 - INFO - 
Step 2: Training model...


✅ Experiment-Setup für 'LSTM' abgeschlossen. Run ID: 2025-09-03_031206_7400_train
📁 Ergebnisordner: C:\Users\ericg\Documents\Mechatronik M Sc\6. Semster\MA\Dev_Ma\ML_Edge_Device\Output\LSTM\2025-09-03_031206_7400_train
--- Starte 3D Trainings-Pipeline (Korrigierte Version) ---

Lade Daten im Modus 'train' mit Strategie 'split'...
✔️ Datendatei gefunden: C:\Users\ericg\Documents\Mechatronik M Sc\6. Semster\MA\Dev_Ma\ML_Edge_Device\Input\Input_Data\mqtt_data_filtered.csv
Geladen: 3542 Zeilen aus 'C:\Users\ericg\Documents\Mechatronik M Sc\6. Semster\MA\Dev_Ma\ML_Edge_Device\Input\Input_Data\mqtt_data_filtered.csv' mit Zeitfeatures
✔️ Datendatei gefunden: C:\Users\ericg\Documents\Mechatronik M Sc\6. Semster\MA\Dev_Ma\ML_Edge_Device\Input\Input_Data\mqtt_data_filtered.csv
Geladen: 3542 Zeilen aus 'C:\Users\ericg\Documents\Mechatronik M Sc\6. Semster\MA\Dev_Ma\ML_Edge_Device\Input\Input_Data\mqtt_data_filtered.csv' mit Zeitfeatures
Nach FE und dropna verbleiben 2636 Zeilen für das Training.


...

2025-09-03 03:16:48,245 - INFO - Ordne Features neu an, um Target 'group4-2_s6_volumetricflowrate' an Position 0 zu platzieren.


Geladen: 3542 Zeilen aus 'C:\Users\ericg\Documents\Mechatronik M Sc\6. Semster\MA\Dev_Ma\ML_Edge_Device\Input\Input_Data\mqtt_data_filtered.csv' mit Zeitfeatures
Nach FE und dropna verbleiben 2636 Zeilen für das Training.

Schritt 3: Scaler anpassen...
✅ Haupt-Scaler (scaler) wurde auf ALLEN Spalten angepasst.
✅ Target-Scaler (y_scaler) wurde NUR auf der Target-Spalte angepasst.

Schritt 4: Gesamte Daten skalieren und 3D-Fenster erstellen...

Trainings-Pipeline abgeschlossen. Shapes: X_train: (2609, 20, 46), y_train: (2609, 8)
Erstelle chronologischen Validierungs-Split. Validation Fraction: 0.2
[I 2025-09-03 03:16:48,701] Trial 14 finished with value: 0.9738306066599403 and parameters: {'units1': 32, 'dropout': 0.1238097033460746, 'learning_rate': 0.0007122456045123542, 'batch_size': 32, 'epochs': 34}. Best is trial 1 with value: 0.9519612795006058.


2025-09-03 03:16:49,325 - INFO - --- 🚀 Starting lstm_simple Training Pipeline ---
2025-09-03 03:16:49,325 - INFO - 
Step 1: Preparing training data...
2025-09-03 03:16:49,372 - INFO - Ordne Features neu an, um Target 'group4-2_s6_volumetricflowrate' an Position 0 zu platzieren.
2025-09-03 03:16:49,438 - INFO - Die zu speichernde Feature-Liste für die Inferenz enthält 46 Spalten.
2025-09-03 03:16:49,440 - INFO - Data preparation complete. Features: 46, X_train shape: (2609, 20, 46)
2025-09-03 03:16:49,441 - INFO - 
Step 2: Training model...


✅ Experiment-Setup für 'LSTM' abgeschlossen. Run ID: 2025-09-03_031649_5864_train
📁 Ergebnisordner: C:\Users\ericg\Documents\Mechatronik M Sc\6. Semster\MA\Dev_Ma\ML_Edge_Device\Output\LSTM\2025-09-03_031649_5864_train
--- Starte 3D Trainings-Pipeline (Korrigierte Version) ---

Lade Daten im Modus 'train' mit Strategie 'split'...
✔️ Datendatei gefunden: C:\Users\ericg\Documents\Mechatronik M Sc\6. Semster\MA\Dev_Ma\ML_Edge_Device\Input\Input_Data\mqtt_data_filtered.csv
Geladen: 3542 Zeilen aus 'C:\Users\ericg\Documents\Mechatronik M Sc\6. Semster\MA\Dev_Ma\ML_Edge_Device\Input\Input_Data\mqtt_data_filtered.csv' mit Zeitfeatures
✔️ Datendatei gefunden: C:\Users\ericg\Documents\Mechatronik M Sc\6. Semster\MA\Dev_Ma\ML_Edge_Device\Input\Input_Data\mqtt_data_filtered.csv
Geladen: 3542 Zeilen aus 'C:\Users\ericg\Documents\Mechatronik M Sc\6. Semster\MA\Dev_Ma\ML_Edge_Device\Input\Input_Data\mqtt_data_filtered.csv' mit Zeitfeatures
Nach FE und dropna verbleiben 2636 Zeilen für das Training.


KeyboardInterrupt: 

In [22]:

# ===============================
# 🔎 Testlauf: Alle Algorithmen, Level="simple", 1 Trial
# ===============================

# Minimal-Defaults, falls oben nicht gesetzt
try:
    SEED
except NameError:
    SEED = 42
try:
    DEFAULT_LAGS
except NameError:
    DEFAULT_LAGS = 20
try:
    DEFAULT_H
except NameError:
    DEFAULT_H = 8
try:
    OBJECTIVE
except NameError:
    OBJECTIVE = "mae_avg"
try:
    QUANT_MODES
except NameError:
    QUANT_MODES = ["no-quant"]

print("Starte Testlauf: alle gefundenen Trainer | level=simple | n_trials=1")
df_all = optimize_all_models(levels=("simple",), n_trials=1, seed=SEED)
print("Fertig.")
df_all.head() if df_all is not None else None


[I 2025-09-03 03:06:23,680] A new study created in memory with name: opt_lstm_simple_v8_no_save


Starte Testlauf: alle gefundenen Trainer | level=simple | n_trials=1

=== Optimize: LSTM | LEVEL: SIMPLE ===


  0%|          | 0/1 [00:00<?, ?it/s]

2025-09-03 03:06:23,697 - INFO - --- 🚀 Starting lstm_simple Training Pipeline ---
2025-09-03 03:06:23,697 - INFO - 
Step 1: Preparing training data...
2025-09-03 03:06:23,768 - INFO - Ordne Features neu an, um Target 'group4-2_s6_volumetricflowrate' an Position 0 zu platzieren.
2025-09-03 03:06:23,804 - INFO - Die zu speichernde Feature-Liste für die Inferenz enthält 46 Spalten.
2025-09-03 03:06:23,805 - INFO - Data preparation complete. Features: 46, X_train shape: (2609, 20, 46)
2025-09-03 03:06:23,806 - INFO - 
Step 2: Training model...


✅ Experiment-Setup für 'LSTM' abgeschlossen. Run ID: 2025-09-03_030623_2847_train
📁 Ergebnisordner: C:\Users\ericg\Documents\Mechatronik M Sc\6. Semster\MA\Dev_Ma\ML_Edge_Device\Output\LSTM\2025-09-03_030623_2847_train
--- Starte 3D Trainings-Pipeline (Korrigierte Version) ---

Lade Daten im Modus 'train' mit Strategie 'split'...
✔️ Datendatei gefunden: C:\Users\ericg\Documents\Mechatronik M Sc\6. Semster\MA\Dev_Ma\ML_Edge_Device\Input\Input_Data\mqtt_data_filtered.csv
Geladen: 3542 Zeilen aus 'C:\Users\ericg\Documents\Mechatronik M Sc\6. Semster\MA\Dev_Ma\ML_Edge_Device\Input\Input_Data\mqtt_data_filtered.csv' mit Zeitfeatures
✔️ Datendatei gefunden: C:\Users\ericg\Documents\Mechatronik M Sc\6. Semster\MA\Dev_Ma\ML_Edge_Device\Input\Input_Data\mqtt_data_filtered.csv
Geladen: 3542 Zeilen aus 'C:\Users\ericg\Documents\Mechatronik M Sc\6. Semster\MA\Dev_Ma\ML_Edge_Device\Input\Input_Data\mqtt_data_filtered.csv' mit Zeitfeatures
Nach FE und dropna verbleiben 2636 Zeilen für das Training.


...

2025-09-03 03:07:17,806 - INFO - Die zu speichernde Feature-Liste für die Inferenz enthält 46 Spalten.
2025-09-03 03:07:17,807 - INFO - Data preparation complete. Features: 46, X_train shape: (2628, 46)
2025-09-03 03:07:17,808 - INFO - 
Step 2: Training model...


Geladen: 3542 Zeilen aus 'C:\Users\ericg\Documents\Mechatronik M Sc\6. Semster\MA\Dev_Ma\ML_Edge_Device\Input\Input_Data\mqtt_data_filtered.csv' mit Zeitfeatures

Schritt 3: Richte Features (X_t) auf zukünftige Zielvariable (Y_{t+1...t+h}) aus...
Nach Ausrichtung und NaN-Filterung verbleiben 2628 Trainingspunkte.

Schritt 4: Skalierung und finale Formatierung...
Passe Feature-Scaler (scaler) auf Trainingsdaten an...
Passe Target-Scaler (y_scaler) auf Trainingsdaten an...

Trainings-Pipeline abgeschlossen. Shapes: X_train: (2628, 46), y_train: (2628, 8)


2025-09-03 03:07:18,844 - INFO - ✅ Model training completed in 1.04 seconds.
2025-09-03 03:07:18,845 - INFO - 
Step 3: Returning trained artifacts without saving.
2025-09-03 03:07:18,846 - INFO - 
✅ Training pipeline finished successfully.


--- Starte 2D Trainings-Pipeline (Finale Prognose-Version) ---

Lade Daten im Modus 'train' mit Strategie 'split'...
✔️ Datendatei gefunden: C:\Users\ericg\Documents\Mechatronik M Sc\6. Semster\MA\Dev_Ma\ML_Edge_Device\Input\Input_Data\mqtt_data_filtered.csv
Geladen: 3542 Zeilen aus 'C:\Users\ericg\Documents\Mechatronik M Sc\6. Semster\MA\Dev_Ma\ML_Edge_Device\Input\Input_Data\mqtt_data_filtered.csv' mit Zeitfeatures
✔️ Datendatei gefunden: C:\Users\ericg\Documents\Mechatronik M Sc\6. Semster\MA\Dev_Ma\ML_Edge_Device\Input\Input_Data\mqtt_data_filtered.csv
Geladen: 3542 Zeilen aus 'C:\Users\ericg\Documents\Mechatronik M Sc\6. Semster\MA\Dev_Ma\ML_Edge_Device\Input\Input_Data\mqtt_data_filtered.csv' mit Zeitfeatures

Schritt 3: Richte Features (X_t) auf zukünftige Zielvariable (Y_{t+1...t+h}) aus...
Nach Ausrichtung und NaN-Filterung verbleiben 2628 Trainingspunkte.

Schritt 4: Skalierung und finale Formatierung...
Passe Feature-Scaler (scaler) auf Trainingsdaten an...
Passe Target-Scal

,algorithm,level,number,value,state,params_batch_size,params_dropout,params_epochs,params_learning_rate,params_units1,...,params_colsample_bytree,params_gamma,params_min_child_weight,params_reg_alpha,params_reg_lambda,params_subsample,params_alpha,params_C,params_epsilon,params_max_iter
0,lstm,simple,0,0.960785,COMPLETE,32.0,0.142607,18.0,0.001379,48.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,cnn1d,simple,0,0.847929,COMPLETE,NaN,0.109799,NaN,0.001147,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,random_forest,simple,0,0.523143,COMPLETE,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,xgboost,simple,0,1.255204,COMPLETE,NaN,NaN,NaN,0.082761,NaN,...,0.831204,1.20223,2.0,0.866176,0.464669,0.919732,NaN,NaN,NaN,NaN
4,light_xgboost,simple,0,0.863461,COMPLETE,NaN,NaN,NaN,0.082761,NaN,...,0.831204,1.20223,2.0,0.866176,0.464669,0.919732,NaN,NaN,NaN,NaN
